# Source-exact structure segmentation

CHUNK partitions the existing normalised page text with document-level structural
evidence and native Docling `HybridChunker`. The stored text is always an exact
source substring; every source character remains within its original page.

Use the **Python (structure)** kernel configured in the [README](../README.md).
Its pinned Docling packages run separately from PDF extraction and GraphRAG.
This notebook consumes prepared raw-page and normalised-page records; it performs
no new PDF extraction or model requests. Its default inputs are saved records of
the public RFC 2795 PDF. See thesis Section 2.8 and Table 6 for the CHUNK method.

In [1]:
import os
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"docling_core\.")
warnings.filterwarnings("ignore", message=r"IProgress not found.*", module=r"tqdm\.")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [2]:
import hashlib
import json
from pathlib import Path

import spacy
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from IPython.display import Markdown, display
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.artifacts import verify_artifacts
from rag_comparison.structure import (
    FrozenTokenizer,
    SourceSerializerProvider,
    build_partitions,
    segment_page_record,
)

## Read the prepared source pages

The manifest binds complete normalised chunks, raw extracted page text, page
origins and questions. Raw line breaks support structure detection; normalised
page text is the unchanged segmentation source. For approved private inputs,
select notebook 01's prepared folder in an ignored executed copy.

In [3]:
page_dir = Path("../examples/structure/page_inputs")
output_dir = Path("../local/structure/rfc2795-inputs")
input_manifest = json.loads(
    (page_dir / "run_manifest.json").read_text(encoding="utf-8")
)
assert input_manifest["chunking_policy"] == config.PAGE_CHUNKING_POLICY
paths = verify_artifacts(page_dir, input_manifest)
records = {
    name: [
        json.loads(line)
        for line in paths[name].read_text(encoding="utf-8").splitlines()
        if line
    ]
    for name in ("chunks", "chunk_origins", "raw_pages", "questions")
}
pages, origins = records["chunks"], records["chunk_origins"]
assert [page["chunk_id"] for page in pages] == [
    origin["chunk_id"] for origin in origins
]
assert {(p["document_id"], p["page_number"]) for p in pages} == {
    (p["document_id"], p["page_number"]) for p in records["raw_pages"]
}
print({"source_pages": len(pages), "questions": len(records["questions"])})

{'source_pages': 20, 'questions': 4}


## Configure the native chunker

The frozen adapter counts `o200k_base` tokens; it loads no model weights.
The source serializer preserves item text. spaCy's rule-based German sentencizer
uses the original punctuation settings. The HybridChunker merges eligible peers
with the configured 800-token cap. Structural thresholds remain in central config.

In [4]:
policy = config.STRUCTURE_POLICY
tokenizer = FrozenTokenizer()
sentences = spacy.blank("de")
sentences.add_pipe(
    "sentencizer", config={"punct_chars": policy["SENTENCE_PUNCTUATION"]}
)
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=policy["CAP"],
    merge_peers=True,
    delim=" ",
    serializer_provider=SourceSerializerProvider(),
)
print(
    {
        "policy": policy["POLICY_ID"],
        "encoding": policy["ENCODING"],
        "maximum_tokens": policy["CAP"],
        "overlap": policy["OVERLAP"],
        "library_versions": policy["VERSIONS"],
    }
)

{'policy': 'page-docling-structure-800-o0-v2', 'encoding': 'o200k_base', 'maximum_tokens': 800, 'overlap': 0, 'library_versions': {'docling-core': '2.27.0', 'semchunk': '2.2.2', 'transformers': '4.57.6', 'tiktoken': '0.13.0', 'spacy': '3.8.14'}}


## Detect structural boundaries, pack and rebind

The reusable function maps raw lines to the normalised source, confirms headings
within the same document and identifies contents/history/list structures. Docling
packs these units. Rebinding then restores the exact source intervals and attached
whitespace. Oversized units use the original technical split; no text is rewritten.

In [5]:
partitions, details = build_partitions(
    pages,
    records["raw_pages"],
    chunker=chunker,
    encode=tokenizer.encode,
    sentence_splitter=sentences,
)
assert set(partitions) == {page["chunk_id"] for page in pages}
rows = []
for page in pages:
    spans = partitions[page["chunk_id"]]
    assert spans[0][0] == 0 and spans[-1][1] == len(page["text"])
    assert all(left[1] == right[0] for left, right in zip(spans, spans[1:]))
    assert "".join(page["text"][start:end] for start, end in spans) == page["text"]
    assert all(
        0 < len(tokenizer.encode(page["text"][start:end])) <= policy["CAP"]
        for start, end in spans
    )
    rows.append(
        (page["page_number"], len(spans), len(details[page["chunk_id"]]["headings"]))
    )
display(
    Markdown(
        tabulate(
            rows[:5],
            headers=["Source page", "Segments", "Confirmed headings"],
            tablefmt="github",
        )
    )
)

|   Source page |   Segments |   Confirmed headings |
|---------------|------------|----------------------|
|             1 |          1 |                    0 |
|             2 |          3 |                    3 |
|             3 |          1 |                    0 |
|             4 |          2 |                    1 |
|             5 |          1 |                    1 |

## Preserve source evidence and stable segment IDs

Each segment ID binds policy, source-page chunk ID, Unicode character interval
and source-text hash. Half-open `[start, end)` offsets index the original Python
string. The segment's Evidence Unit remains its source page. Both systems consume
the same segment text and identity, while retaining their own native retrieval.

In [6]:
segments, edges, graph_inputs = [], [], []
for page, origin in zip(pages, origins, strict=True):
    page_segments, page_edges, page_inputs = segment_page_record(
        page,
        origin,
        tokenizer.encode,
        tokenizer.decode,
        character_spans=partitions[page["chunk_id"]],
    )
    segments.extend(page_segments)
    edges.extend(page_edges)
    graph_inputs.extend(page_inputs)
assert len(segments) == len(edges) == len(graph_inputs)
assert (
    [row["segment_id"] for row in segments]
    == [row["target_artifact_id"] for row in edges]
    == [row["input_record_id"] for row in graph_inputs]
)
assert all(
    segment["text"] == graph["text"]
    for segment, graph in zip(segments, graph_inputs, strict=True)
)
print(
    {
        "segments": len(segments),
        "maximum_tokens": max(row["token_count"] for row in segments),
        "source_pages": len(pages),
    }
)

{'segments': 44, 'maximum_tokens': 687, 'source_pages': 20}


## Inspect one page's exact intervals

The preview below selects the page with the most segments. The checks above cover
all pages. Compare each stored segment with its source slice; offsets and token
counts explain the resulting system inputs without displaying the full corpus.

In [7]:
selected_page = max(pages, key=lambda row: len(partitions[row["chunk_id"]]))
selected_segments = [
    row for row in segments if row["source_page_chunk_id"] == selected_page["chunk_id"]
]
display(
    Markdown(
        tabulate(
            [
                (
                    row["segment_index_on_page"],
                    row["character_start"],
                    row["character_end"],
                    row["token_count"],
                    row["text"][:100],
                )
                for row in selected_segments[:5]
            ],
            headers=["Segment", "Start", "End", "Tokens", "Source-text preview"],
            tablefmt="github",
        )
    )
)
for segment in selected_segments:
    assert (
        segment["text"]
        == selected_page["text"][segment["character_start"] : segment["character_end"]]
    )

|   Segment |   Start |   End |   Tokens | Source-text preview                                                                                  |
|-----------|---------|-------|----------|------------------------------------------------------------------------------------------------------|
|         1 |       0 |   506 |      132 | RFC 2795 The Infinite Monkey Protocol Suite (IMPS) 1 April 2000 4. A SIMIAN should respond to a TYPE |
|         2 |     506 |   622 |       28 | 6. A SIMIAN must respond to a TRANSCRIPT request by establishing a CHIMP session to send the transcr |
|         3 |     622 |  1356 |      209 | 5.4 Example ZOO-to-SIMIAN Exchanges using KEEPER Assume a ZOO (SanDiego) must interact with a monkey |
|         4 |    1356 |  1586 |       57 | 6. CHIMP Specification Following is a description of the Cross-Habitat Idiomatic Message Protocol (C |

## Use the same prepared-input contract in both systems

`chunk_id` aliases the stable segment ID for the existing pipeline notebooks.
Source page identity and offsets remain explicit. The origin row maps that
segment to exactly one original page Evidence Unit. Repeated page retrieval
positions remain repeated; segmentation does not create new relevance units.

In [8]:
chunks = [
    {
        **row,
        "chunk_id": row["segment_id"],
        "document_id": row["document_id"],
        "page_number": row["page_number"],
    }
    for row in segments
]
chunk_origins = [
    {
        "chunk_id": chunk["chunk_id"],
        "document_id": chunk["document_id"],
        "page_number": chunk["page_number"],
        "corpus_snapshot_id": chunk["corpus_snapshot_id"],
        "origin_evidence_unit_ids": edge["source_evidence_unit_ids"],
        "projection_status": edge["projection_status"],
    }
    for chunk, edge in zip(chunks, edges, strict=True)
]
assert [row["chunk_id"] for row in chunks] == [row["chunk_id"] for row in chunk_origins]
assert all(len(row["origin_evidence_unit_ids"]) == 1 for row in chunk_origins)

## Save the shared population without Azure requests

The output manifest binds original segment records, source edges, native graph
inputs, pipeline aliases, questions and structure details. It also identifies the
source-page manifest and policy. Repeated execution with unchanged inputs writes
the same records; changed inputs require a distinct output folder.

In [9]:
assert output_dir.resolve().is_relative_to(Path("../local").resolve())
output_dir.mkdir(parents=True, exist_ok=True)
files = {
    "segments": segments,
    "projection_edges": edges,
    "graph_inputs": graph_inputs,
    "chunks": chunks,
    "chunk_origins": chunk_origins,
    "questions": records["questions"],
    "structure_details": [
        {"source_page_chunk_id": key, **value} for key, value in details.items()
    ],
}
artifacts = {}
for role, rows in files.items():
    path = output_dir / f"{role}.jsonl"
    content = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)
    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            "Use a new folder for changed inputs."
        )
    path.write_text(content, encoding="utf-8")
    artifacts[role] = {
        "path": path.name,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }
manifest = {
    "schema_version": "classical-rag-inputs-v1",
    "artifact_kind": "inputs",
    "chunking_policy": policy["POLICY_ID"],
    "question_set_version": input_manifest["question_set_version"],
    "source_page_count": len(pages),
    "segment_count": len(segments),
    "source_manifest_sha256": hashlib.sha256(
        (page_dir / "run_manifest.json").read_bytes()
    ).hexdigest(),
    "structure_policy": policy,
    "artifacts": artifacts,
}
(output_dir / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
verify_artifacts(output_dir, manifest)
print(
    {
        "prepared_segments": len(segments),
        "C-CHUNK/G-CHUNK/C-FINAL_inputs": output_dir.as_posix(),
    }
)

{'prepared_segments': 44, 'C-CHUNK/G-CHUNK/C-FINAL_inputs': '../local/structure/rfc2795-inputs'}
